In [ ]:

import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:

import os

keywords = [
    "train.csv",
    "validation.csv",
    "test.csv",
    "mobilenetv3_fp32.tflite",
    "class_mapping.json",
    "treatment_mapping.json",
    ".keras",
    ".h5"
]

print("========== REQUIRED FILES ==========")

for root, _, files in os.walk("/kaggle/input"):
    for filename in files:
        if any(key in filename.lower() for key in keywords):
            print(os.path.join(root, filename))


In [ ]:

import pandas as pd
from pathlib import Path

DATASET_ROOT = Path(
    "/kaggle/input/datasets/emm arex/plantdisease/PlantVillage"
)

TRAIN_CSV = (
    "/kaggle/input/datasets/ankaprian/"
    "diseaseleafclassifier-metadata/train.csv"
)
VALID_CSV = (
    "/kaggle/input/datasets/ankaprian/"
    "diseaseleafclassifier-metadata/validation.csv"
)
TEST_CSV = (
    "/kaggle/input/datasets/ankaprian/"
    "diseaseleafclassifier-metadata/test.csv"
)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VALID_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nTest CSV columns:")
print(test_df.columns.tolist())

print("\nFirst 5 test rows:")
display(test_df.head())


In [ ]:

from pathlib import Path

# PlantVillage image directory
DATASET_ROOT = Path(
    "/kaggle/input/datasets/emmarex/plantdisease/"
    "plantvillage/PlantVillage"
)

print("Dataset root exists:", DATASET_ROOT.exists())

# Check whether CSV filenames match actual image files
print("\nChecking test image paths...")

missing = []
found = 0

for _, row in test_df.iterrows():
    image_path = (
        DATASET_ROOT
        / row["class_name"]
        / row["image_name"]
    )

    if image_path.exists():
        found += 1
    else:
        missing.append(str(image_path))

print("Test images found:", found)
print("Test images missing:", len(missing))

if missing:
    print("\nFirst 5 missing paths:")
    for path in missing[:5]:
        print(path)


In [ ]:

import tensorflow as tf
import numpy as np
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from pathlib import Path

FP32_PATH = (
    "/kaggle/input/datasets/ankaprian/"
    "mobilenetv3-fp32/mobilenetv3_fp32.tflite"
)

# Load the existing FP32 model
interpreter = tf.lite.Interpreter(model_path=FP32_PATH)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

print("Input shape:", input_info["shape"])
print("Input dtype:", input_info["dtype"])

y_true = []
y_pred = []

for _, row in test_df.iterrows():
    image_path = (
        DATASET_ROOT / row["class_name"] / row["image_name"]
    )

    # Resize and preserve pixel range 0–255
    image = Image.open(image_path).convert("RGB")
    image = image.resize((224, 224))
    image_array = np.asarray(image, dtype=np.float32)
    image_array = np.expand_dims(image_array, axis=0)

    interpreter.set_tensor(input_info["index"], image_array)
    interpreter.invoke()

    prediction = interpreter.get_tensor(output_info["index"])[0]
    predicted_index = int(np.argmax(prediction))

    y_true.append(row["class_name"])
    y_pred.append(predicted_index)

# Load the index-to-class mapping
import json

CLASS_MAP_PATH = (
    "/kaggle/input/datasets/ankaprian/"
    "class-mapping-json/class_mapping.json"
)

with open(CLASS_MAP_PATH, "r", encoding="utf-8") as f:
    class_mapping = json.load(f)

# Convert predicted indices to disease names
y_pred_names = [class_mapping[str(i)] for i in y_pred]

print("\n========== FP32 TEST RESULTS ==========")
print("Test images:", len(y_true))
print(f"Accuracy : {accuracy_score(y_true, y_pred_names):.4f}")
print(f"Precision: {precision_score(y_true, y_pred_names, average='weighted', zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true, y_pred_names, average='weighted', zero_division=0):.4f}")
print(f"F1 Score : {f1_score(y_true, y_pred_names, average='weighted', zero_division=0):.4f}")


In [ ]:

import json
import numpy as np

with open(CLASS_MAP_PATH, "r", encoding="utf-8") as f:
    class_mapping = json.load(f)

print("Number of model output classes:",
      interpreter.get_output_details()[0]["shape"][-1])

print("Number of mapped classes:", len(class_mapping))

print("\nClass mapping:")
for index in sorted(class_mapping, key=lambda x: int(x)):
    print(index, "->", class_mapping[index])

# Check whether the mapping agrees with the CSV labels
csv_classes = set(test_df["class_name"])
mapped_classes = set(class_mapping.values())

print("\nClasses in test CSV:", len(csv_classes))
print("Classes in mapping:", len(mapped_classes))
print("Missing from mapping:", sorted(csv_classes - mapped_classes))
print("Extra in mapping:", sorted(mapped_classes - csv_classes))


In [ ]:

# ==========================================
# REBUILD DATA PIPELINE
# Based on the original notebook
# ==========================================

import os
import numpy as np
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 15

# 1. Index all PlantVillage images
image_paths = {}

for root, _, files in os.walk(str(DATASET_ROOT)):
    for filename in files:
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths[filename] = os.path.join(root, filename)

print("Indexed images:", len(image_paths))

# 2. Add image paths to each split
train_df["filepath"] = train_df["image_name"].map(image_paths)
val_df["filepath"] = val_df["image_name"].map(image_paths)
test_df["filepath"] = test_df["image_name"].map(image_paths)

print("Missing train paths:", train_df["filepath"].isna().sum())
print("Missing validation paths:", val_df["filepath"].isna().sum())
print("Missing test paths:", test_df["filepath"].isna().sum())

# Stop if any image is missing
assert train_df["filepath"].notna().all()
assert val_df["filepath"].notna().all()
assert test_df["filepath"].notna().all()

# 3. Reproduce the original alphabetical class mapping
class_names = sorted(train_df["class_name"].unique())

class_to_index = {
    name: index for index, name in enumerate(class_names)
}

index_to_class = {
    index: name for name, index in class_to_index.items()
}

assert len(class_names) == NUM_CLASSES

train_df["label"] = train_df["class_name"].map(class_to_index)
val_df["label"] = val_df["class_name"].map(class_to_index)
test_df["label"] = test_df["class_name"].map(class_to_index)

print("\nNumber of classes:", len(class_names))
print("Class mapping:")
for index, name in index_to_class.items():
    print(index, "->", name)

# 4. Load images in the same format as the original notebook
def load_image(filepath, label):
    image = tf.io.read_file(filepath)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    return image, label

# 5. Build datasets
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["filepath"].values, train_df["label"].values)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["filepath"].values, val_df["label"].values)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["filepath"].values, test_df["label"].values)
)

train_ds = (
    train_ds
    .shuffle(len(train_df), seed=42)
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# 6. Verify a batch
images, labels = next(iter(train_ds))

print("\nTraining dataset ready.")
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
NUM_CLASSES = 15

# Data augmentation — same as the original notebook
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05
    )
], name="data_augmentation")

# MobileNetV3Small feature extractor
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=True
)

# Initially freeze the pretrained layers
base_model.trainable = False

# Build the classification model
inputs = tf.keras.Input(
    shape=(224, 224, 3),
    name="input_image"
)

x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="predictions"
)(x)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="DiseaseLeafClassifier_MobileNetV3Small"
)

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Loss: Sparse categorical crossentropy")

In [ ]:
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Save the best model during training
checkpoint_path = "/kaggle/working/best_mobilenetv3_baseline.keras"

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

# Stop if validation loss stops improving
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate when progress stalls
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

callbacks = [
    checkpoint,
    early_stopping,
    reduce_lr
]

print("Training callbacks configured successfully.")

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df["label"].values
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in enumerate(class_weights_array)
}

print("Class weights calculated successfully.")
print(class_weights)

In [ ]:
history_baseline = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

In [ ]:
# Unfreeze the pretrained backbone
base_model.trainable = True

# Freeze earlier layers; fine-tune only the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep Batch Normalization layers frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile with a much smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Fine-tuning model prepared.")
print("Trainable parameters:",
      sum(tf.keras.backend.count_params(w)
          for w in model.trainable_weights))

In [ ]:
fine_tune_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_mobilenetv3_finetuned.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=fine_tune_callbacks
)

In [ ]:
# Load the best checkpoint saved during fine-tuning
best_model = tf.keras.models.load_model(
    "/kaggle/working/best_mobilenetv3_finetuned.keras"
)

# Save a clearly named final Keras model
final_model_path = "/kaggle/working/final_mobilenetv3_finetuned.keras"
best_model.save(final_model_path)

print("Best fine-tuned model loaded successfully.")
print("Saved model:", final_model_path)
print("Input shape:", best_model.input_shape)
print("Output shape:", best_model.output_shape)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import numpy as np

# Load the best fine-tuned model
best_model = tf.keras.models.load_model(
    "/kaggle/working/final_mobilenetv3_finetuned.keras"
)

# Generate predictions on the test dataset
y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

y_prob = best_model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

# Calculate evaluation metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(
    y_true, y_pred, average="weighted", zero_division=0
)
recall = recall_score(
    y_true, y_pred, average="weighted", zero_division=0
)
f1 = f1_score(
    y_true, y_pred, average="weighted", zero_division=0
)

print(f"Test Accuracy:  {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Weighted Precision: {precision:.4f}")
print(f"Weighted Recall:    {recall:.4f}")
print(f"Weighted F1-score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=class_names,
    zero_division=0
))

In [ ]:
import tensorflow as tf
import os

keras_model_path = (
    "/kaggle/working/final_mobilenetv3_finetuned.keras"
)
fp32_tflite_path = (
    "/kaggle/working/mobilenetv3_finetuned_fp32.tflite"
)

# Load the saved fine-tuned Keras model
model_for_conversion = tf.keras.models.load_model(
    keras_model_path
)

# Convert to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(
    model_for_conversion
)

fp32_tflite_model = converter.convert()

# Save the converted model
with open(fp32_tflite_path, "wb") as f:
    f.write(fp32_tflite_model)

model_size_mb = os.path.getsize(fp32_tflite_path) / (1024 * 1024)

print("FP32 TFLite conversion successful.")
print(f"Model path: {fp32_tflite_path}")
print(f"Model size: {model_size_mb:.2f} MB")

# Verify model input/output details
interpreter = tf.lite.Interpreter(
    model_path=fp32_tflite_path
)
interpreter.allocate_tensors()

print("Input details:", interpreter.get_input_details()[0])
print("Output details:", interpreter.get_output_details()[0])

In [ ]:
interpreter = tf.lite.Interpreter(
    model_path="/kaggle/working/mobilenetv3_finetuned_fp32.tflite"
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("INPUT TENSOR")
for item in input_details:
    print("Name:", item["name"])
    print("Shape:", item["shape"])
    print("Dtype:", item["dtype"])
    print("Quantization:", item["quantization"])

print("\nOUTPUT TENSOR")
for item in output_details:
    print("Name:", item["name"])
    print("Shape:", item["shape"])
    print("Dtype:", item["dtype"])
    print("Quantization:", item["quantization"])

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score

interpreter = tf.lite.Interpreter(
    model_path="/kaggle/working/mobilenetv3_finetuned_fp32.tflite"
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

y_true = []
y_pred_tflite = []

for images, labels in test_ds:
    images_np = images.numpy()
    labels_np = labels.numpy()

    for image, label in zip(images_np, labels_np):
        input_tensor = np.expand_dims(image, axis=0).astype(np.float32)

        interpreter.set_tensor(input_details["index"], input_tensor)
        interpreter.invoke()

        output = interpreter.get_tensor(output_details["index"])
        prediction = int(np.argmax(output[0]))

        y_true.append(int(label))
        y_pred_tflite.append(prediction)

tflite_accuracy = accuracy_score(y_true, y_pred_tflite)

print(f"FP32 TFLite Test Accuracy: {tflite_accuracy:.4%}")
print(f"Evaluated images: {len(y_true)}")

In [ ]:
# Representative dataset for INT8 calibration
def representative_dataset():
    for images, _ in train_ds.take(100):
        for image in images:
            image = tf.cast(image, tf.float32)
            yield [tf.expand_dims(image, axis=0).numpy()]

# Verify that representative samples are available
sample_count = 0

for sample in representative_dataset():
    if sample_count == 0:
        print("Representative input shape:", sample[0].shape)
        print("Representative input dtype:", sample[0].dtype)

    sample_count += 1

print("Representative samples prepared:", sample_count)

In [ ]:
import tensorflow as tf
import os

keras_model_path = (
    "/kaggle/working/final_mobilenetv3_finetuned.keras"
)
int8_model_path = (
    "/kaggle/working/mobilenetv3_finetuned_int8.tflite"
)

# Load the trained model
model_for_int8 = tf.keras.models.load_model(keras_model_path)

# Configure the TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(
    model_for_int8
)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset

# Require integer-only operations
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

# Require integer input and output tensors
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# Convert and save
int8_model = converter.convert()

with open(int8_model_path, "wb") as f:
    f.write(int8_model)

size_mb = os.path.getsize(int8_model_path) / (1024 * 1024)

print("INT8 conversion successful.")
print("Saved to:", int8_model_path)
print(f"INT8 model size: {size_mb:.2f} MB")

In [ ]:
import tensorflow as tf
import os

int8_path = "/kaggle/working/mobilenetv3_finetuned_int8.tflite"

# Read the model file without creating an interpreter
with open(int8_path, "rb") as f:
    model_content = f.read()

print("Model file size:", len(model_content), "bytes")

# Inspect the model using TensorFlow Lite's analyzer
tf.lite.experimental.Analyzer.analyze(
    model_content=model_content
)

In [ ]:
import tensorflow as tf

int8_path = "/kaggle/working/mobilenetv3_finetuned_int8.tflite"

# Disable default delegates such as XNNPACK
interpreter_int8 = tf.lite.Interpreter(
    model_path=int8_path,
    experimental_op_resolver_type=(
        tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
)

interpreter_int8.allocate_tensors()

input_details = interpreter_int8.get_input_details()[0]
output_details = interpreter_int8.get_output_details()[0]

print("Allocation successful!")
print("Input dtype:", input_details["dtype"])
print("Input quantization:", input_details["quantization"])
print("Output dtype:", output_details["dtype"])
print("Output quantization:", output_details["quantization"])

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

int8_path = "/kaggle/working/mobilenetv3_finetuned_int8.tflite"

interpreter_int8 = tf.lite.Interpreter(
    model_path=int8_path,
    experimental_op_resolver_type=(
        tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
)
interpreter_int8.allocate_tensors()

input_details = interpreter_int8.get_input_details()[0]
output_details = interpreter_int8.get_output_details()[0]

input_scale, input_zero_point = input_details["quantization"]
output_scale, output_zero_point = output_details["quantization"]

y_true_int8 = []
y_pred_int8 = []

for images, labels in test_ds:
    for image, label in zip(images.numpy(), labels.numpy()):
        image = np.expand_dims(image, axis=0).astype(np.float32)

        # Quantize the input using the model's actual scale and zero point
        image_int8 = np.clip(
            np.round(image / input_scale + input_zero_point),
            -128,
            127
        ).astype(np.int8)

        interpreter_int8.set_tensor(
            input_details["index"], image_int8
        )
        interpreter_int8.invoke()

        output_int8 = interpreter_int8.get_tensor(
            output_details["index"]
        )

        # Dequantize output scores to floating point
        output_float = (
            output_int8.astype(np.float32) - output_zero_point
        ) * output_scale

        y_pred_int8.append(int(np.argmax(output_float[0])))
        y_true_int8.append(int(label))

print(f"INT8 Test Accuracy: {accuracy_score(y_true_int8, y_pred_int8):.4%}")
print(f"Weighted Precision: {precision_score(y_true_int8, y_pred_int8, average='weighted', zero_division=0):.4f}")
print(f"Weighted Recall:    {recall_score(y_true_int8, y_pred_int8, average='weighted', zero_division=0):.4f}")
print(f"Weighted F1-score:  {f1_score(y_true_int8, y_pred_int8, average='weighted', zero_division=0):.4f}")
print("Evaluated images:", len(y_true_int8))

 Next step: Check whether the model is truly integer-quantized

In [ ]:
import tensorflow as tf
from collections import Counter

int8_path = "/kaggle/working/mobilenetv3_finetuned_int8.tflite"

interpreter = tf.lite.Interpreter(
    model_path=int8_path,
    experimental_op_resolver_type=(
        tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
)
interpreter.allocate_tensors()

details = interpreter.get_tensor_details()

dtype_counts = Counter(
    str(t["dtype"]) for t in details if t["dtype"] is not None
)

print("Tensor dtype counts:")
for dtype, count in dtype_counts.items():
    print(f"{dtype}: {count}")

print("\nInput quantization:", interpreter.get_input_details()[0]["quantization"])
print("Output quantization:", interpreter.get_output_details()[0]["quantization"])

print("\nSample quantized tensors:")
shown = 0
for tensor in details:
    scale, zero_point = tensor["quantization"]
    if len(scale) > 0 if hasattr(scale, "__len__") else scale != 0:
        print(
            tensor["name"],
            "| dtype:", tensor["dtype"],
            "| scale:", scale,
            "| zero point:", zero_point
        )
        shown += 1
        if shown >= 10:
            break

Compare FP32 and INT8 predictions on the same images

In [ ]:
import numpy as np
import tensorflow as tf

fp32_path = "/kaggle/working/mobilenetv3_finetuned_fp32.tflite"
int8_path = "/kaggle/working/mobilenetv3_finetuned_int8.tflite"

# FP32 interpreter
fp32 = tf.lite.Interpreter(model_path=fp32_path)
fp32.allocate_tensors()
fp_in = fp32.get_input_details()[0]
fp_out = fp32.get_output_details()[0]

# INT8 interpreter without default delegates
int8 = tf.lite.Interpreter(
    model_path=int8_path,
    experimental_op_resolver_type=(
        tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
)
int8.allocate_tensors()
q_in = int8.get_input_details()[0]
q_out = int8.get_output_details()[0]

in_scale, in_zero = q_in["quantization"]
out_scale, out_zero = q_out["quantization"]

correct_fp32 = 0
correct_int8 = 0
same_prediction = 0
total = 0

for images, labels in test_ds.take(3):
    for image, label in zip(images.numpy(), labels.numpy()):
        image = image[None, ...].astype(np.float32)

        # FP32 prediction
        fp32.set_tensor(fp_in["index"], image)
        fp32.invoke()
        fp_pred = int(np.argmax(fp32.get_tensor(fp_out["index"])[0]))

        # Quantize input and run INT8
        q_image = np.clip(
            np.round(image / in_scale + in_zero), -128, 127
        ).astype(np.int8)

        int8.set_tensor(q_in["index"], q_image)
        int8.invoke()

        q_output = int8.get_tensor(q_out["index"])[0]
        dequantized = (q_output.astype(np.float32) - out_zero) * out_scale
        q_pred = int(np.argmax(dequantized))

        label = int(label)
        correct_fp32 += fp_pred == label
        correct_int8 += q_pred == label
        same_prediction += fp_pred == q_pred
        total += 1

print("Images checked:", total)
print(f"FP32 accuracy on sample: {correct_fp32 / total:.2%}")
print(f"INT8 accuracy on sample: {correct_int8 / total:.2%}")
print(f"FP32 and INT8 predictions agree: {same_prediction / total:.2%}")

verify the input quantization range

In [ ]:
import numpy as np
import tensorflow as tf

int8_path = "/kaggle/working/mobilenetv3_finetuned_int8.tflite"

interpreter = tf.lite.Interpreter(
    model_path=int8_path,
    experimental_op_resolver_type=(
        tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
scale, zero_point = input_info["quantization"]

# Inspect a batch of real test images
images, labels = next(iter(test_ds))
images = images.numpy()

quantized_images = np.clip(
    np.round(images / scale + zero_point),
    -128,
    127
).astype(np.int8)

print("Original image range:",
      float(images.min()), "to", float(images.max()))
print("Input quantization scale:", scale)
print("Input zero point:", zero_point)
print("Quantized image range:",
      int(quantized_images.min()), "to",
      int(quantized_images.max()))

# Recover approximate float values to measure quantization error
reconstructed = (
    quantized_images.astype(np.float32) - zero_point
) * scale

print("Mean absolute input reconstruction error:",
      float(np.mean(np.abs(images - reconstructed))))

test a better-calibrated INT8 conversion

In [ ]:
import tensorflow as tf
import numpy as np
import os

keras_path = "/kaggle/working/final_mobilenetv3_finetuned.keras"
int8_calibrated_path = (
    "/kaggle/working/mobilenetv3_finetuned_int8_calibrated.tflite"
)

calibration_images = []

# Collect up to 500 real training images
for images, _ in train_ds.take(16):
    for image in images.numpy():
        calibration_images.append(image.astype(np.float32))
        if len(calibration_images) >= 500:
            break
    if len(calibration_images) >= 500:
        break

print("Calibration images:", len(calibration_images))
print("Calibration image shape:", calibration_images[0].shape)

def representative_dataset_v2():
    for image in calibration_images:
        yield [image[np.newaxis, ...]]

model_for_conversion = tf.keras.models.load_model(keras_path)

converter = tf.lite.TFLiteConverter.from_keras_model(
    model_for_conversion
)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_v2
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

calibrated_int8_model = converter.convert()

with open(int8_calibrated_path, "wb") as f:
    f.write(calibrated_int8_model)

print("Calibrated INT8 model saved:", int8_calibrated_path)
print(
    "Model size (MB):",
    round(os.path.getsize(int8_calibrated_path) / (1024 * 1024), 2)
)

evaluate the newly calibrated INT8 model

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

int8_path = "/kaggle/working/mobilenetv3_finetuned_int8_calibrated.tflite"

interpreter = tf.lite.Interpreter(
    model_path=int8_path,
    experimental_op_resolver_type=(
        tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
)
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

input_scale, input_zero = inp["quantization"]
output_scale, output_zero = out["quantization"]

y_true, y_pred = [], []

for images, labels in test_ds:
    for image, label in zip(images.numpy(), labels.numpy()):
        image = image[None, ...].astype(np.float32)

        q_image = np.clip(
            np.round(image / input_scale + input_zero),
            -128, 127
        ).astype(np.int8)

        interpreter.set_tensor(inp["index"], q_image)
        interpreter.invoke()

        q_output = interpreter.get_tensor(out["index"])[0]
        scores = (q_output.astype(np.float32) - output_zero) * output_scale

        y_true.append(int(label))
        y_pred.append(int(np.argmax(scores)))

print(f"Calibrated INT8 accuracy: {accuracy_score(y_true, y_pred):.4%}")
print(f"Weighted precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Weighted recall:    {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Weighted F1-score:  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")

In [ ]:
import tensorflow as tf
import os

models = {
    "Original INT8":
        "/kaggle/working/mobilenetv3_finetuned_int8.tflite",
    "Calibrated INT8":
        "/kaggle/working/mobilenetv3_finetuned_int8_calibrated.tflite"
}

for name, path in models.items():
    print(f"\n--- {name} ---")
    print("Exists:", os.path.exists(path))

    interpreter = tf.lite.Interpreter(
        model_path=path,
        experimental_op_resolver_type=(
            tf.lite.experimental.OpResolverType
            .BUILTIN_WITHOUT_DEFAULT_DELEGATES
        )
    )
    interpreter.allocate_tensors()

    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]

    print("Input dtype:", inp["dtype"])
    print("Input quantization:", inp["quantization"])
    print("Output dtype:", out["dtype"])
    print("Output quantization:", out["quantization"])

Compare the FP32 and INT8 models' confidence on the same images

In [ ]:
import numpy as np
import tensorflow as tf

fp32 = tf.lite.Interpreter(
    model_path="/kaggle/working/mobilenetv3_finetuned_fp32.tflite"
)
fp32.allocate_tensors()

int8 = tf.lite.Interpreter(
    model_path="/kaggle/working/mobilenetv3_finetuned_int8.tflite",
    experimental_op_resolver_type=(
        tf.lite.experimental.OpResolverType
        .BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
)
int8.allocate_tensors()

fp_in = fp32.get_input_details()[0]
fp_out = fp32.get_output_details()[0]
q_in = int8.get_input_details()[0]
q_out = int8.get_output_details()[0]

in_scale, in_zero = q_in["quantization"]
out_scale, out_zero = q_out["quantization"]

agreements = []
fp_confidences = []
int8_confidences = []

for images, _ in test_ds.take(3):
    for image in images.numpy():
        image = image[None, ...].astype(np.float32)

        fp32.set_tensor(fp_in["index"], image)
        fp32.invoke()
        fp_scores = fp32.get_tensor(fp_out["index"])[0]

        q_image = np.clip(
            np.round(image / in_scale + in_zero),
            -128, 127
        ).astype(np.int8)

        int8.set_tensor(q_in["index"], q_image)
        int8.invoke()

        q_scores = int8.get_tensor(q_out["index"])[0]
        q_scores = (q_scores.astype(np.float32) - out_zero) * out_scale

        agreements.append(
            int(np.argmax(fp_scores) == np.argmax(q_scores))
        )
        fp_confidences.append(float(np.max(fp_scores)))
        int8_confidences.append(float(np.max(q_scores)))

print("Images compared:", len(agreements))
print(f"Prediction agreement: {np.mean(agreements):.2%}")
print(f"Mean FP32 maximum score: {np.mean(fp_confidences):.4f}")
print(f"Mean INT8 maximum score: {np.mean(int8_confidences):.4f}")

#try FP16 quantization

In [ ]:
import tensorflow as tf
import os

keras_path = "/kaggle/working/final_mobilenetv3_finetuned.keras"
fp16_path = "/kaggle/working/mobilenetv3_finetuned_fp16.tflite"

model_fp16 = tf.keras.models.load_model(keras_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model_fp16)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

fp16_model = converter.convert()

with open(fp16_path, "wb") as f:
    f.write(fp16_model)

print("FP16 conversion successful.")
print("Saved to:", fp16_path)
print(
    "FP16 model size:",
    round(os.path.getsize(fp16_path) / (1024 * 1024), 2),
    "MB"
)

Evaluate FP16 accuracy

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

fp16_path = "/kaggle/working/mobilenetv3_finetuned_fp16.tflite"

interpreter_fp16 = tf.lite.Interpreter(
    model_path=fp16_path
)
interpreter_fp16.allocate_tensors()

input_details = interpreter_fp16.get_input_details()[0]
output_details = interpreter_fp16.get_output_details()[0]

print("Input dtype:", input_details["dtype"])
print("Output dtype:", output_details["dtype"])

y_true_fp16 = []
y_pred_fp16 = []

for images, labels in test_ds:
    for image, label in zip(images.numpy(), labels.numpy()):
        input_tensor = image[np.newaxis, ...].astype(np.float32)

        interpreter_fp16.set_tensor(
            input_details["index"], input_tensor
        )
        interpreter_fp16.invoke()

        scores = interpreter_fp16.get_tensor(
            output_details["index"]
        )[0]

        y_true_fp16.append(int(label))
        y_pred_fp16.append(int(np.argmax(scores)))

print(
    f"FP16 Test Accuracy: "
    f"{accuracy_score(y_true_fp16, y_pred_fp16):.4%}"
)
print(
    f"Weighted Precision: "
    f"{precision_score(y_true_fp16, y_pred_fp16, average='weighted', zero_division=0):.4f}"
)
print(
    f"Weighted Recall: "
    f"{recall_score(y_true_fp16, y_pred_fp16, average='weighted', zero_division=0):.4f}"
)
print(
    f"Weighted F1-score: "
    f"{f1_score(y_true_fp16, y_pred_fp16, average='weighted', zero_division=0):.4f}"
)
print("Evaluated images:", len(y_true_fp16))

Measure inference latency

In [ ]:
import tensorflow as tf
import numpy as np
import time
import os

model_paths = {
    "FP32": "/kaggle/working/mobilenetv3_finetuned_fp32.tflite",
    "FP16": "/kaggle/working/mobilenetv3_finetuned_fp16.tflite"
}

# Use a small, fixed sample for a fair comparison
benchmark_images = []
for images, _ in test_ds.take(2):
    benchmark_images.extend(images.numpy())

benchmark_images = benchmark_images[:64]

def benchmark_model(model_path, images, warmup=10, repeats=3):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_info = interpreter.get_input_details()[0]
    output_info = interpreter.get_output_details()[0]

    # Warm-up
    for image in images[:warmup]:
        interpreter.set_tensor(
            input_info["index"],
            image[None, ...].astype(np.float32)
        )
        interpreter.invoke()

    timings = []

    for _ in range(repeats):
        start = time.perf_counter()

        for image in images:
            interpreter.set_tensor(
                input_info["index"],
                image[None, ...].astype(np.float32)
            )
            interpreter.invoke()

        timings.append(
            (time.perf_counter() - start) / len(images) * 1000
        )

    return np.median(timings), os.path.getsize(model_path) / (1024 * 1024)

for name, path in model_paths.items():
    latency_ms, size_mb = benchmark_model(path, benchmark_images)
    print(
        f"{name}: {size_mb:.2f} MB | "
        f"Median inference latency: {latency_ms:.2f} ms/image"
    )

Save the FP16 model as a deployment artifact

In [ ]:
import os
import json
import tensorflow as tf

model_path = "/kaggle/working/mobilenetv3_finetuned_fp16.tflite"
metadata_path = "/kaggle/working/deployment_metadata.json"

assert os.path.exists(model_path), "FP16 model file not found."

interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

metadata = {
    "model_name": "DiseaseLeafClassifier_MobileNetV3Small",
    "format": "TensorFlow Lite",
    "optimization": "FP16 weight quantization",
    "input_shape": input_info["shape"].tolist(),
    "input_dtype": str(input_info["dtype"]),
    "output_shape": output_info["shape"].tolist(),
    "output_dtype": str(output_info["dtype"]),
    "num_classes": 15,
    "model_size_mb": round(os.path.getsize(model_path) / (1024 * 1024), 2),
    "test_accuracy": 0.932138,
    "classes": class_names
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Deployment model verified.")
print("Model:", model_path)
print("Metadata:", metadata_path)
print("Input:", metadata["input_shape"], metadata["input_dtype"])
print("Output:", metadata["output_shape"], metadata["output_dtype"])

In [ ]:
import json
import os

results = {
    "project": "DiseaseLeafClassifier",
    "architecture": "MobileNetV3Small",
    "test_images": 2063,
    "models": {
        "FP32": {
            "accuracy_percent": 93.1653,
            "size_mb": 3.60,
            "latency_ms_per_image": 2.36
        },
        "FP16": {
            "accuracy_percent": 93.2138,
            "precision": 0.9354,
            "recall": 0.9321,
            "f1_score": 0.9324,
            "size_mb": 1.84,
            "latency_ms_per_image": 2.43
        },
        "INT8": {
            "accuracy_percent": 69.6074,
            "precision": 0.8206,
            "recall": 0.6961,
            "f1_score": 0.7089,
            "size_mb": 1.18
        },
        "INT8_calibrated": {
            "accuracy_percent": 57.4891,
            "precision": 0.8037,
            "recall": 0.5749,
            "f1_score": 0.5750,
            "size_mb": 1.18
        }
    },
    "recommended_candidate": "FP16",
    "deployment_status": "Pending Raspberry Pi testing",
    "latency_environment": "Kaggle benchmark; not Raspberry Pi latency"
}

results_path = "/kaggle/working/model_evaluation_results.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print("Results report saved:", results_path)